In [1]:
# imports!
from datasets import load_dataset, Dataset
import pandas as pd
import os
import ast
import torch
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

In [2]:
models = ["meta-llama/Llama-3.1-8B", "Qwen/Qwen3-8B", "mistralai/Mistral-7B-v0.3"]
directories = ["llama_hellaswag", "qwen_hellaswag", "mistral_hellaswag"]
adapters = ["./llama_hellaswag_adapter", "./qwen_hellaswag_adapter", "./mistral_hellaswag_adapter"]
num = 0

model_id = models[num]
directory = directories[num]
adapter = adapters[num]

In [3]:
# choose number of sentences
num_sen = 1000

# helper function
def make_zero_shot_prompt(row, dataset):
    if dataset == "hellaswag":
        question = row["ctx"].strip()
        choices = row["endings"]

    elif dataset == "mmlu":
        question = row["question"].strip()
        choices = row["choices"]

    labels = ["A", "B", "C", "D"]
    choices_text = "\n".join(f"{label}. {choice}" for label, choice in zip(labels, choices))

    return f"{question}\n{choices_text}\nAnswer:"

# add prompt used for testing
def no_shot_prompt(df, dataset):
    df["text"] = df.apply(
        lambda row: make_zero_shot_prompt(row, dataset),
        axis=1,
    )
    return df

In [4]:
## == TRAINING IMPORTS == ##

# import hellaswag
hellaswag_dataset = load_dataset("Rowan/hellaswag")
hellaswag_df = pd.DataFrame(hellaswag_dataset['train']).sample(n=num_sen, random_state=8)

# store
hellaswag_df = no_shot_prompt(hellaswag_df, "hellaswag")
hellaswag_df.to_csv("~/TDA_RI/TDA_reason-interpret/train/hellaswag_train.csv", index=False)

In [5]:
hellaswag_df = hellaswag_df.reset_index(drop=True)
hellaswag_df['text'][0]

'[header] How to have an effective handshake [title] Know when to use your handshake. [step] The appropriate times to shake another person\'s hand include : [substeps] When you are introduced to someone when you say goodbye to someone at the beginning or the end of a business, social, church, or other meeting whenever it seems appropriate within a business context, such as sealing a deal. [title] Be the first to extend your hand.\nA. [step] It can be instinctive when you first meet someone. But taking the initiative to shake them is a great way to show a familiarity with the person as well as keep things interesting.\nB. [step] Shake the other person\'s hand and say hello. Say hi, smile, or even shake their hand before allowing your hand to be extended.\nC. [step] This makes a strong, lasting impression on the person at the receiving end. It is also about control; by offering your hand first, you are leading the way.\nD. [step] Nod " hello " to others as they walk past you and stick yo

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [7]:
config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        # "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)

In [8]:
args = TrainingArguments(
    output_dir=directory,
    num_train_epochs=4, # replace this, depending on your dataset
    per_device_train_batch_size=16,
    learning_rate=1e-5,
    optim="sgd"
)

In [9]:
# trainer = SFTTrainer(
#     model=model,
#     args=args,
#     train_dataset=hellaswag_df,
#     dataset_text_field='text',
#     max_seq_length=1024,
# )

# trainer.train()

hellaswag_dataset = Dataset.from_pandas(hellaswag_df, preserve_index=False)

args = SFTConfig(
    output_dir="./results",
    max_length=1024,
    dataset_text_field="text",
    loss_type="nll",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=hellaswag_dataset,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,2.693511
20,2.467265
30,2.396532
40,2.336281
50,2.295097
60,2.280733
70,2.252813
80,2.284165
90,2.197817
100,2.181189


[W911 17:33:30.845816457 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1285554176 bytes (free: 119734272, total: 20982726656).


TrainOutput(global_step=375, training_loss=2.200670669555664, metrics={'train_runtime': 1307.6061, 'train_samples_per_second': 2.294, 'train_steps_per_second': 0.287, 'total_flos': 3.60889841492951e+16, 'train_loss': 2.200670669555664, 'entropy': 2.1492064952850343, 'mean_token_accuracy': 0.5232162952423096, 'num_tokens': 560400.0, 'epoch': 3.0})

In [10]:
# adapter_model = trainer.model
# merged_model = adapter_model.merge_and_unload()

# trained_tokenizer = trainer.tokenizer

adapter_model = trainer.model

merged_model = adapter_model.merge_and_unload()

trained_tokenizer = trainer.processing_class

# model.push_to_hub(model_id) # the tokenizer will stay the same
merged_model.save_pretrained(directory)
trained_tokenizer.save_pretrained(directory)

/home/kim/TDA_RI/TDA_reason-interpret/venv/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('llama_hellaswag/tokenizer_config.json', 'llama_hellaswag/tokenizer.json')

In [11]:
# model_id = "mistral_hellaswag"

# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)

# config = model.config
# del config.quantization_config
# if hasattr(config, "_pre_quantization_dtype"):
#     del config._pre_quantization_dtype
# model.config = config

# model.dequantize()

base_model_id = model_id
adapter_path = adapter

trainer.model.save_pretrained(adapter_path)
trainer.processing_class.save_pretrained(adapter_path)

# Load the ORIGINAL base model in BF16, not 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)

# Attach your trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

# Merge LoRA into the BF16 base model
merged_model = model.merge_and_unload()

merged_model.push_to_hub(directory)
tokenizer.push_to_hub(directory)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
/home/kim/TDA_RI/TDA_reason-interpret/venv/lib/python3.10/site-packages/accelerate/utils/modeling.py:1583: UserWarning: Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
/home/kim/TDA_RI/TDA_reason-interpret/venv/lib/python3.10/site-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_pro

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/kimlopez/llama_hellaswag/commit/5578d67419c6f15fe8192e482e69b5ce6626f78c', commit_message='Upload tokenizer', commit_description='', oid='5578d67419c6f15fe8192e482e69b5ce6626f78c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kimlopez/llama_hellaswag', endpoint='https://huggingface.co', repo_type='model', repo_id='kimlopez/llama_hellaswag'), pr_revision=None, pr_num=None)